In [ ]:
from dash import Dash, dcc, html
from dash.dependencies import Output, Input
import plotly.graph_objs as go
from collections import deque
import numpy as np

app = Dash(__name__)
app.server  # Deployment

# Parameters
seed = [3,4,1]
memory = 13
k = 0.35  # Damp constant
delta_psi = 1.0  # Initial Δψ

past, present, future = seed
byte_stream = deque([seed[-1]], maxlen=512)
analog_surface = deque([0], maxlen=512)
history = deque(seed, maxlen=memory)
x_vals = deque([0], maxlen=256)
counter = 1
h_ratios = deque([], maxlen=256)
q_vals = deque([], maxlen=256)
dpsi_vals = deque([delta_psi], maxlen=256)

@app.callback(
    Output('live-graph', 'figure'),
    Input('interval-component', 'n_intervals')
)
def update_graph(n):
    global past, present, future, counter, delta_psi

    # Recursive fold
    delta1 = abs(past + present) % 10
    delta2 = abs(present + future) % 10
    harmonic = (delta1 + delta2 + abs(past - future)) % 10
    byte_stream.append(harmonic)
    history.append(harmonic)

    # Analog
    analog_val = np.mean(history)
    analog_surface.append(analog_val if round(analog_val) == 5 else 0.35)

    # H-ratio
    h = np.sum(byte_stream) / np.sum(analog_surface) if np.sum(analog_surface) > 0 else 0
    h_ratios.append(h)

    # Q(H)
    q = 1 - abs(analog_val / len(history) - 0.35)
    q_vals.append(q)

    # Δψ update
    delta_psi = (1 - k) * delta_psi + 1 / (1 + np.exp(-delta_psi))
    dpsi_vals.append(delta_psi)

    past, present, future = present, future, harmonic
    x_vals.append(counter)
    counter += 1

    # Traces
    trace1 = go.Scatter(x=list(x_vals), y=list(byte_stream), mode='lines', name='Byte Pulse', line=dict(color='royalblue'))
    trace2 = go.Scatter(x=list(x_vals), y=list(analog_surface), mode='lines', name='Analog Surface', line=dict(color='darkorange'))
    trace3 = go.Scatter(x=list(x_vals), y=list(h_ratios), mode='lines', name='H-Ratio', line=dict(color='green'))
    trace4 = go.Scatter(x=list(x_vals), y=list(q_vals), mode='lines', name='Q(H)', line=dict(color='purple'))
    trace5 = go.Scatter(x=list(x_vals), y=list(dpsi_vals), mode='lines', name='Δψ', line=dict(color='red'))

    layout = go.Layout(
        xaxis=dict(title='Time'),
        yaxis=dict(title='Value', range=[0, 10]),
        margin=dict(l=40, r=20, t=40, b=10),
        legend=dict(x=0, y=1),
        hovermode='closest'
    )

    # Genesis snap check
    if q > 0.95 and abs(h - 0.35) < 0.05 and delta_psi < 0.01:
        print("Genesis Snap: Conditions met at t=", counter)  # Console log for snap

    return {'data': [trace1, trace2, trace3, trace4, trace5], 'layout': layout}

app.run_server(debug=False)